# Phase 3: Model Building & Evaluation
## SaaS SLA Breach Prediction Project

### Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    roc_auc_score, roc_curve, precision_score, recall_score, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
warnings.filterwarnings('ignore')
print('✓ All libraries imported successfully!')

### Step 2: Load Processed Data

In [ ]:
df = pd.read_csv("../data/processed/cleaned_tickets.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()[:10]}...")

### Step 3: Feature Engineering

In [ ]:
# Convert datetime columns
df['started'] = pd.to_datetime(df['started'], errors='coerce')
df['ended'] = pd.to_datetime(df['ended'], errors='coerce')

# Calculate resolution time in hours
df['resolution_time_hours'] = (df['ended'] - df['started']).dt.total_seconds() / 3600

# Define SLA breach (breach if resolution_time > 48 hours)
SLA_THRESHOLD = 48
df['sla_breach'] = (df['resolution_time_hours'] > SLA_THRESHOLD).astype(int)

print(f"SLA Breach distribution:")
print(df['sla_breach'].value_counts())

### Step 4: Data Preparation for Modeling

In [ ]:
# Select numeric features
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
if 'sla_breach' in numeric_features:
    numeric_features.remove('sla_breach')

# Sample 30% for faster training
df_sample = df.sample(frac=0.3, random_state=42)
X = df_sample[numeric_features].copy()
# Fill NaN with mean for each column
for col in X.columns:
    X[col].fillna(X[col].mean(), inplace=True)
# Remove any rows with remaining NaNs
X = X.dropna()
y = df_sample.loc[X.index, 'sla_breach']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"✓ Using 30% of data ({len(df_sample):,} samples) for faster training")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled successfully")

### Step 5: Model Training

In [ ]:
print("Training Logistic Regression...")
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_accuracy = accuracy_score(y_test, lr_pred)
print(f"✓ Logistic Regression Accuracy: {lr_accuracy:.4f}")

In [ ]:
print("Training Random Forest (20 trees)...")
rf_model = RandomForestClassifier(n_estimators=20, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)
print(f"✓ Random Forest Accuracy: {rf_accuracy:.4f}")

In [ ]:
print("Training XGBoost (20 trees)...")
xgb_model = XGBClassifier(n_estimators=20, max_depth=5, learning_rate=0.1, random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test, xgb_pred)
print(f"✓ XGBoost Accuracy: {xgb_accuracy:.4f}")

### Step 6: Model Evaluation

In [ ]:
models = {
    'Logistic Regression': (lr_model, lr_pred, 'scaled'),
    'Random Forest': (rf_model, rf_pred, 'normal'),
    'XGBoost': (xgb_model, xgb_pred, 'normal')
}

print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
for name, (model, predictions, scaler_type) in models.items():
    accuracy = accuracy_score(y_test, predictions)
    print(f"\n{name}: {accuracy:.4f}")
    print(confusion_matrix(y_test, predictions))

In [ ]:
best_model_name = max(models.items(), key=lambda x: accuracy_score(y_test, x[1][1]))[0]
best_model, best_pred, _ = models[best_model_name]

print(f"\n{'='*60}")
print(f"BEST MODEL: {best_model_name}")
print(f"{'='*60}")
print("\nClassification Report:")
print(classification_report(y_test, best_pred))

### Step 7: ROC-AUC Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for name, (model, _, scaler_type) in models.items():
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test_scaled if scaler_type == 'scaled' else X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc = roc_auc_score(y_test, y_proba)
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves - Model Comparison', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Step 8: Save Best Model

In [ ]:
model_path = '../artifacts/best_model.pkl'
pickle.dump(best_model, open(model_path, 'wb'))
print(f"✓ Best model saved to {model_path}")

### Step 9: Model Performance Summary with Bar Graph

In [ ]:
# Calculate all metrics for performance comparison
performance_metrics = {}

for name, (model, pred, _) in models.items():
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred, zero_division=0)
    recall = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)
    
    performance_metrics[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

In [ ]:
# Create performance bar graph
fig, ax = plt.subplots(figsize=(14, 7))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.25
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for idx, (model_name, scores) in enumerate(performance_metrics.items()):
    values = [scores[metric] for metric in metrics]
    ax.bar(x + idx*width, values, width, label=model_name, color=colors[idx], alpha=0.85)

ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Comparison - All Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.legend(fontsize=11, loc='lower right')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 1.0])

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', label_type='edge', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*70)
print("FINAL MODEL PERFORMANCE SCORES")
print("="*70)
for model_name, scores in performance_metrics.items():
    print(f"\n{model_name}:")
    for metric, score in scores.items():
        print(f"  {metric:15s}: {score:.4f}")

print(f"\n{'='*70}")
print(f"🎯 BEST MODEL: {best_model_name}")
print(f"✓ Best Model Accuracy: {performance_metrics[best_model_name]['Accuracy']:.4f}")
print(f"{'='*70}")